# 03 — Differential abundance (DESeq2, R vs NR)

Runs every DESeq2 responder vs non-responder comparison used in the paper, with one set of settings.

**Settings** (`run_deseq()`): size factors → keep genes with mean normalized count (baseMean) > 5 → `DESeq()` → `results(alpha = 0.1)`.
Design `~ response` with levels `NR`, `R`, so log2FoldChange = R vs NR (positive = higher in responders).

**Inputs** (input_dir): `counts/counts_raw_cart_paper.tsv.gz`, `metadata/sample_metadata.tsv`, `reference/gencode.biotype.name.key.tsv.gz`.

**Outputs** (`results/deseq/`, full tables with columns Row.names, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj, gene_name, gene_type):

| file | samples | used by |
|---|---|---|
| `discovery_C1C2_all_genes.csv` | Cohorts 1+2 (n = 56); padj < 0.1 gives the 776 DAGs | Fig4 volcano, heatmaps, SFig18 |
| `cohort3_all_genes.csv` | Cohort 3 (validation) | SFig15 fold-change concordance |
| `all_samples_all_genes.csv` | all 91 samples | SFig17 sensitivity |
| `subset_12mo_all_genes.csv` | ≥12-month cohort (n = 85) | SFig17 sensitivity |

In [ ]:
source("../../utils/setup.R")
suppressPackageStartupMessages(library(DESeq2))

## Load inputs

In [ ]:
meta_paper <- load_sample_metadata(order = "paper")        # default sample order
meta_clin  <- load_sample_metadata(order = "clinical")     # sample order of the 12-month analysis
counts_raw <- load_counts("raw")
gene_key   <- load_gene_key()

## DESeq2 helper

In [ ]:
# DESeq2 R vs NR on `samples` (rows of load_sample_metadata(); response is a factor with levels NR, R).
# Genes with mean size-factor-normalized count <= 5 are removed before DESeq().
run_deseq <- function(counts, samples, min_base_mean = 5, alpha = 0.1) {
  col_data <- data.frame(response = factor(samples$response, levels = RESPONSE_LEVELS),
                         row.names = samples$sample_id)
  dds <- DESeqDataSetFromMatrix(countData = counts[, samples$sample_id],
                                colData = col_data, design = ~ response)
  dds <- estimateSizeFactors(dds)
  dds <- dds[rowMeans(counts(dds, normalized = TRUE)) > min_base_mean, ]
  dds <- DESeq(dds, quiet = TRUE)
  res <- results(dds, alpha = alpha)
  summary(res)
  # merge() sorts by Row.names
  merge(as.data.frame(res), gene_key, by.x = 0, by.y = "gene_id")
}

write_deseq <- function(res_df, name) {
  write.csv(res_df, res_path("deseq", paste0(name, "_all_genes.csv")), row.names = FALSE)
  cat(name, ": ", nrow(res_df), " genes tested, ", sum(res_df$padj < 0.1, na.rm = TRUE),
      " with padj < 0.1\n", sep = "")
}

## Discovery: Cohorts 1+2 (Figure 4)

In [ ]:
samples_disc <- meta_paper[meta_paper$cohort != "Cohort 3", ]
table(samples_disc$response)
res_disc <- run_deseq(counts_raw, samples_disc)
write_deseq(res_disc, "discovery_C1C2")

## Validation: Cohort 3 (Supplementary Figure 15)

In [ ]:
samples_c3 <- meta_paper[meta_paper$cohort == "Cohort 3", ]
table(samples_c3$response)
res_c3 <- run_deseq(counts_raw, samples_c3)
write_deseq(res_c3, "cohort3")

In [ ]:
# Fold-change concordance on the discovery DAGs (padj < 0.1); genes filtered out of the
# Cohort 3 run are dropped by the inner join.
fc <- inner_join(res_disc[, c("Row.names", "log2FoldChange", "padj")],
                 res_c3[, c("Row.names", "log2FoldChange", "padj")],
                 by = "Row.names", suffix = c(".disc", ".val")) %>%
  filter(!is.na(padj.disc), padj.disc < 0.1)
cat("Discovery DAGs also tested in Cohort 3:", nrow(fc), "\n")
cor.test(fc$log2FoldChange.disc, fc$log2FoldChange.val, method = "pearson")

## Sensitivity: all samples and the ≥12-month cohort (Supplementary Figure 17)

In [ ]:
samples_all  <- meta_clin
samples_12mo <- filter_12mo(meta_clin)
c(all = nrow(samples_all), mo12 = nrow(samples_12mo))

res_all <- run_deseq(counts_raw, samples_all)
write_deseq(res_all, "all_samples")

res_12mo <- run_deseq(counts_raw, samples_12mo)
write_deseq(res_12mo, "subset_12mo")